<a href="https://colab.research.google.com/github/alumond/Activity-1/blob/main/FInetune_LLm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load the Data

In [ ]:
from datasets import load_dataset
from huggingface_hub import notebook_login

notebook_login()
afrimedqa = load_dataset("intronhealth/afrimedqa_v2")


Inspect the Dataset Structure

In [ ]:
print(afrimedqa)

DatasetDict({
    train: Dataset({
        features: ['sample_id', 'split', 'gender', 'discipline', 'clinical_experience', 'country', 'question_type', 'prompt', 'question', 'question_clean', 'answer_options', 'correct_answer', 'answer_rationale', 'specialty', 'region_specific', 'mentions_Africa', 'mentions_age', 'mentions_gender', 'tier', 'neg_percent', 'quality', 'rated_african', 'rated_correct', 'rated_omission', 'rated_hallucination', 'rated_reasonable', 'rated_harmful', 'rated_bias', 'version'],
        num_rows: 15275
    })
})


View Column Names

In [ ]:
print(afrimedqa['train'].column_names)

['sample_id', 'split', 'gender', 'discipline', 'clinical_experience', 'country', 'question_type', 'prompt', 'question', 'question_clean', 'answer_options', 'correct_answer', 'answer_rationale', 'specialty', 'region_specific', 'mentions_Africa', 'mentions_age', 'mentions_gender', 'tier', 'neg_percent', 'quality', 'rated_african', 'rated_correct', 'rated_omission', 'rated_hallucination', 'rated_reasonable', 'rated_harmful', 'rated_bias', 'version']


Examine a Sample

In [ ]:
print(afrimedqa['train'][0])

{'sample_id': 'ff784abc6b461a6a15d7f8bf5e6560d3393474fe4748699ff4c7f2124292b95f', 'split': 'train', 'gender': 'Male', 'discipline': 'Non-Health', 'clinical_experience': 'Intern', 'country': 'NG', 'question_type': 'consumer_queries', 'prompt': 'Your female friend complains of weight gain, high blood pressure and thinks she has Lupus Nephritis and is going to visit the nearest doctor. Question: ', 'question': 'Can you provide recommendations for managing stress and improving mental well-being in relation to my condition?', 'question_clean': 'Can you provide recommendations for managing stress and improving mental well-being in relation to my condition?', 'answer_options': None, 'correct_answer': None, 'answer_rationale': None, 'specialty': None, 'region_specific': None, 'mentions_Africa': None, 'mentions_age': None, 'mentions_gender': None, 'tier': 'crowdsourced', 'neg_percent': 0.0, 'quality': None, 'rated_african': None, 'rated_correct': None, 'rated_omission': None, 'rated_hallucinati

#Data Processing

In [ ]:
def format_data_for_llm(example):
    """
    Formats a single row of the AfriMed-QA dataset into a standardized
    instruction-response template for LLM fine-tuning.

    :param example: A single row (dictionary) from the dataset.
    :return: A dictionary containing the single 'text' key with the formatted string.
    """
    q_type = example['question_type']
    specialty = example['specialty']

    # --- 1. Construct the Instruction ---
    # Set the model persona and context
    instruction = (
        f"You are an expert African health AI. Answer the following {q_type} in the "
        f"context of African healthcare, focusing on the {specialty} specialty. "
        f"Provide a comprehensive, accurate, and regionally appropriate response."
    )

    # Add the Question
    instruction += f"\n\nQuestion: {example['question']}"

    # Add Options only for Multiple Choice Questions (MCQ)
    if q_type == 'MCQ' and example['answer_options'] is not None:
        # Simple cleanup of options string for presentation
        options = example['answer_options'].replace("'", "")
        instruction += f"\nOptions: {options}"

    # --- 2. Construct the Final Training String ---
    # The final format separates the Instruction (Input) from the Response (Target Output)
    formatted_text = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n{example['answer_rationale']}"
    )

    return {"text": formatted_text}

# --- How to Apply the Code ---

# 1. Apply the mapping function to the 'train' split
# This creates a new column 'text' with the formatted string for every row.
afrimedqa_formatted = afrimedqa['train'].map(
    format_data_for_llm,
    # Remove all original columns to save memory, keeping only the new 'text' column
    remove_columns=afrimedqa['train'].column_names
)

# 2. Inspect the first example to verify the output
print("--- Example of the formatted training data ---")
print(afrimedqa_formatted[0]['text'])

--- Example of the formatted training data ---
### Instruction:
You are an expert African health AI. Answer the following consumer_queries in the context of African healthcare, focusing on the None specialty. Provide a comprehensive, accurate, and regionally appropriate response.

Question: Can you provide recommendations for managing stress and improving mental well-being in relation to my condition?

### Response:
None


#1. Clean the Null Values

A. Remove or Replace Null Responses

In [ ]:
# Assuming your formatted dataset is named afrimedqa_formatted
print(f"Original size: {len(afrimedqa_formatted)}")

# Filter out rows where the 'text' column ends with the corrupted response
afrimedqa_clean = afrimedqa_formatted.filter(
    lambda example: not example['text'].endswith("### Response:\nNone")
)

print(f"Cleaned size: {len(afrimedqa_clean)}")

Original size: 15275
Cleaned size: 1710


#2. Tokenization

A. Load Tokenizer

In [ ]:
from huggingface_hub import notebook_login

# This will prompt you to enter your Hugging Face token.
# Your token can be found in your Hugging Face account settings under 'Access Tokens'.
notebook_login()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# The correct, authenticated model ID
model_id = "meta-llama/Llama-3.2-3B-Instruct"

# Ensure you have accepted the license on the model page first!
# https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16, # Recommended for Llama 3 models
    device_map="auto",
    # No need to pass 'token' argument!
)

print(f"Successfully loaded model: {model_id}")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Successfully loaded model: meta-llama/Llama-3.2-3B-Instruct


In [ ]:
from transformers import AutoTokenizer

# Replace 'your/base-llm-model' with the actual model name (e.g., 'meta-llama/Llama-2-7b-hf')
model_id = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Set the pad_token for models that don't have one (e.g., Llama)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
# ... other imports

In [ ]:
pip install -U bitsandbytes

Model, Data Loading, and Formatting (FP16/QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset
import os

model_id = "meta-llama/Llama-3.2-3B"

# --- A. Load Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- B. Load Model in 4-bit Quantization (QLoRA) - FP16 VERSION ---
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    # Using float16 (FP16) compute dtype for compatibility
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

# Load the model with FP16 dtype
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=quantization_config
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
print("Model and Tokenizer loaded successfully, using FP16 compute and QLoRA.")

# --- C. Data Loading and Formatting ---

print("\nLoading and formatting mock AfriMed-QA dataset...")
# Mock data creation (increased size)
data = {
    'question_type': ['MCQ', 'SAQ', 'consumer_queries', 'SAQ'] * 100, # Increased size
    'question': ["A 35-year-old female presents with fever and cough for 5 days. What's the diagnosis?", "Describe malaria management.", "I feel tired. What's wrong?", "Outline pediatric HIV treatment."] * 100, # Increased size
    'answer_options': ["['A. TB', 'B. CAP']", None, None, None] * 100, # Increased size
    'answer_rationale': ["The most likely diagnosis is CAP.", "Management includes Artesunate.", "See a doctor immediately.", "Treatment involves anti-retroviral therapy."] * 100, # Increased size
    'specialty': ['Internal Medicine', 'Pediatrics', 'General Practice', 'Pediatrics'] * 100, # Increased size
}
afrimedqa = Dataset.from_dict(data)

def format_data_for_llm(example):
    q_type = example['question_type']
    specialty = example['specialty']
    instruction = (
        f"You are an expert African health AI. Answer the following {q_type} in the "
        f"context of African healthcare, focusing on the {specialty} specialty. "
        f"Provide a comprehensive, accurate, and regionally appropriate response."
    )
    if q_type == 'MCQ' and example.get('answer_options') is not None:
        options = example['answer_options'].strip("[]").replace("'", "")
        instruction += f"\n\nQuestion: {example['question']}\nOptions: {options}"
    else:
        instruction += f"\n\nQuestion: {example['question']}"
    formatted_text = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n{example['answer_rationale']}"
    )
    return {"text": formatted_text}

afrimedqa_formatted = afrimedqa.map(format_data_for_llm, remove_columns=afrimedqa.column_names)
print(f"Dataset formatted and ready. Total examples: {len(afrimedqa_formatted)}")

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Model and Tokenizer loaded successfully, using FP16 compute and QLoRA.

Loading and formatting mock AfriMed-QA dataset...


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset formatted and ready. Total examples: 400


LoRA and Training Arguments (FP16 & Low VRAM)

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments

# LoRA configuration (unchanged)
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

output_dir = "./afrimed_llama3_sft_output"
os.makedirs(output_dir, exist_ok=True)

# Training Arguments (Using FP16 and reduced batch size)
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,         # Reduced VRAM
    gradient_accumulation_steps=8,         # Effective batch size of 16
    optim="paged_adamw_8bit",
    logging_steps=25,
    save_steps=200,
    learning_rate=2e-4,
    fp16=True,                             # CRITICAL FIX: Enable FP16
    bf16=False,                            # Disable BF16
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
)
print("Training Arguments configured with FP16 and low VRAM settings.")

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511
Training Arguments configured with FP16 and low VRAM settings.


Tokenization and Launch Training

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=2048)

tokenized_dataset = afrimedqa_formatted.map(
    tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=afrimedqa_formatted.column_names
)
print("Dataset tokenization complete.")

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Starting Supervised Fine-Tuning... Good luck! 🚀")
# Launch the training process
trainer.train()

print("\n\nTraining Complete! LoRA adapter weights are saved in the output directory.")

Map (num_proc=4):   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset tokenization complete.


/tmp/ipython-input-914324257.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Starting Supervised Fine-Tuning... Good luck! 🚀


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: almond-owolabi01 (almond-owolabi01-stanforte-edge) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
25,0.883500
50,0.026600
75,0.021600




Training Complete! LoRA adapter weights are saved in the output directory.


Save the Final LoRA Adapter


In [ ]:
# The LoRA adapter weights are saved to the output directory
final_output_dir = "./afrimed_llama3_sft_output/final_checkpoint"

# Save the final adapter weights
trainer.model.save_pretrained(final_output_dir)
tokenizer.save_pretrained(final_output_dir)

print(f"Final LoRA adapter and tokenizer saved to: {final_output_dir}")

Final LoRA adapter and tokenizer saved to: ./afrimed_llama3_sft_output/final_checkpoint


Run an Inference Test

In [ ]:
from transformers import GenerationConfig
import torch
import re # Import regex for cleaning the output

# --- CRITICAL FIX 1: Explicitly Set the Llama 3 Chat Template ---
LLAMA_3_TEMPLATE = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
        "{{ '<|start_header_id|>system<|end_header_id|>\n' + message['content'] + '<|eot_id|>' }}"
    "{% elif message['role'] == 'user' %}"
        "{{ '<|start_header_id|>user<|end_header_id|>\n' + message['content'] + '<|eot_id|>' }}"
    "{% elif message['role'] == 'assistant' %}"
        "{{ '<|start_header_id|>assistant<|end_header_id|>\n' + message['content'] + '<|eot_id|>' }}"
    "{% endif %}"
    "{% if loop.last and add_generation_prompt %}"
        "{{ '<|start_header_id|>assistant<|end_header_id|>\n' }}"
    "{% endif %}"
    "{% endfor %}"
)
tokenizer.chat_template = LLAMA_3_TEMPLATE
# ----------------------------------------------------

# --- CRITICAL FIX 2: Disable Gradient Checkpointing for fast inference ---
model.config.use_cache = True
model.gradient_checkpointing_disable()

# Base Instruction for the model (The "System" message)
system_instruction = (
    "You are an expert African health AI. Answer the medical question "
    "concisely and accurately, focusing on African healthcare context. "
    "Do not include internal section headings like 'Outline' or 'Options'."
)

# User's current query (The "User" message)
user_query = (
    "A 6-year-old child presents with severe vomiting and confusion in rural Ghana. "
    "What is the immediate treatment?"
)

# Combine into the Llama 3 chat format
chat_history = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": user_query},
]

# --- CRITICAL FIX 3: Manually convert the token list to a tensor (Indexing Fix) ---
# 1. Apply the template but do NOT return tensors yet.
input_tokens_list = tokenizer.apply_chat_template(
    chat_history,
    tokenize=True,
    add_generation_prompt=True
)

# 2. Convert the list to a 2D PyTorch tensor [1, sequence_length]
input_ids = torch.tensor([input_tokens_list], dtype=torch.long).to(model.device)

# 3. Create the attention mask manually
attention_mask = torch.ones_like(input_ids).to(model.device)
# -----------------------------------------------------------------------


# 1. Generate the response
generation_config = GenerationConfig(
    max_new_tokens=256,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    repetition_penalty=1.2
)

print("\n--- Generating Fine-Tuned Response (Attempt 8: Final Syntax Check) ---")
with torch.no_grad():
    output = model.generate(
        input_ids,
        attention_mask=attention_mask, # Pass the mask explicitly
        generation_config=generation_config
    )

# 2. Decode and print the output
full_output_text = tokenizer.decode(output[0], skip_special_tokens=False)

# Clean the output based on the Llama 3 tags
assistant_tag = "<|start_header_id|>assistant<|end_header_id|>"
response_start_index = full_output_text.rfind(assistant_tag)
# Ensure this line is complete and correct:
clean_response = full_output_text[response_start_index + len(assistant_tag):].strip().replace("<|eot_id|>", "")

# --- Aggressively clean up known garbage tokens ---
clean_response = re.split(r'(gnoreaddComponentDictionary|Question:|### Response:)', clean_response, 1)[0]
clean_response = clean_response.strip()

print(clean_response)


--- Generating Fine-Tuned Response (Attempt 8: Final Syntax Check) ---
Treatment involves intravenous fluid replacement. Seek emergency pediatric HIV treatment immediately.


# Task
Build a chatbot using the fine-tuned Llama 3 model and the provided code.

## Load base model and tokenizer

### Subtask:
Load the original Llama 3 base model and tokenizer that you used for fine-tuning.


**Reasoning**:
The subtask is to load the original Llama 3 base model and tokenizer. This requires importing necessary classes, defining the model ID, and loading both the tokenizer and the model, setting the appropriate data type and device mapping.

